# Module 13 Guided Lab: Classification Models

## Logistic Regression and Decision Trees

**BAN 6003: Data Management and Analytics Integration**

In Module 12, we used linear regression to predict a continuous outcome: arrival delay in minutes. This week, we move to classification.

The main idea is:

> Classification models predict categories or classes.

For this lab, we will use `nycflights13` to predict **whether a flight arrives at least 15 minutes late**.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Distinguish regression problems from classification problems.
2. Create a binary classification target and check class balance.
3. Use training, validation, and test sets for different purposes.
4. Fit a logistic regression model and turn probabilities into classes with a threshold.
5. Fit a shallow decision tree classifier and read its first split.
6. Review a confusion matrix: false alarms and missed late flights.
7. Calculate accuracy, precision, and recall, and explain why accuracy can mislead.
8. Compare models and choose a threshold that fits a business limit.
9. Apply classification to a second business problem: rank customers for a call campaign and explain the result.

This is an introductory classification lab. The goal is to learn the workflow and to interpret the results carefully, not to build the perfect model.

## Business Scenario

An airline operations team wants to **flag flights that are likely to arrive late**.

> Will this flight arrive at least 15 minutes late?

- **Target:** `arr_late_15`
- **Class 1:** the flight arrived 15 or more minutes late
- **Class 0:** the flight did not arrive 15 or more minutes late

As in Module 12, **what you can predict depends on WHEN you predict**. The team has two possible uses:

| Model | Decision time | Business use | Features |
|---|---|---|---|
| **Model 1** (main lab) | Right after departure | **In-flight alert:** warn passengers with tight connections and the arrival airport | Schedule, route, airline, weather at the scheduled hour, plane age, **plus** the actual departure delay |
| **Model 2** (your Final Practice) | The evening before | **Staffing alert:** plan extra staff and proactive customer messages | The same features **without** departure delay |

## Important Timing Note

Every feature must be known at the time the prediction is made.

- `dep_delay` is known **right after departure**, so Model 1 may use it, but Model 2 may not.
- `air_time` (minutes in the air) is known **only after landing**. By then, we already know whether the flight was late. So we **never use `air_time`** in this lab: it would be data leakage for both decisions.
- Our weather is the weather that was **observed** in the scheduled hour. The evening before, the airline would only have a forecast, so Model 2's results are, if anything, a little optimistic.

## 0. Setup

### Package setup note

You can work in GitHub Codespaces or in local VS Code. On the UTSA campus network Codespaces may not connect; use the local setup there (see the Canvas page on local setup).

In **GitHub Codespaces**, the required packages are already installed. You do not need to run any `%pip install` command.

On your **local computer**, if an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.

In [ ]:
# Codespaces: nycflights13 and scikit-learn are already installed.
# Local only: if the imports below fail, uncomment and run the needed line(s):
# %pip install nycflights13 "setuptools<81"
# (setuptools<81 is needed because nycflights13 uses pkg_resources, which newer setuptools removed.)
# %pip install -q scikit-learn

import nycflights13
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

# Find the repository folder (works from the repo root or from notebooks/)
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Folder where the lab saves its CSV files
output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Build the Flight ABT

We build the same flight-level ABT as in Module 12. One row represents one flight.

- **`flights`**: flights with a known departure delay and arrival delay (cancelled and diverted flights have no arrival delay).
- **`weather`**: weather at the origin airport in the scheduled departure hour (duplicate weather keys removed before the join).
- **`planes`**: `plane_age = 2013 - plane_year`.
- Day of the week.

We use the same reproducible sample of 60,000 flights (`random_state=42`) so every cell runs quickly.

In [ ]:
flights = nycflights13.flights
weather = nycflights13.weather
planes = nycflights13.planes

# Flights with a known departure delay and arrival delay
flight_rows = flights.dropna(subset=["dep_delay", "arr_delay"]).copy()

# One weather row per airport and hour (remove duplicate keys before joining)
weather_keys = ["origin", "year", "month", "day", "hour"]
weather_hourly = weather.drop_duplicates(subset=weather_keys)
weather_hourly = weather_hourly[weather_keys + ["precip", "visib", "wind_speed", "temp"]]

# The year each plane was built
plane_info = planes[["tailnum", "year"]].rename(columns={"year": "plane_year"})

# Add weather and plane year to each flight (left joins keep every flight)
flight_abt = (
    flight_rows
    .merge(weather_hourly, on=weather_keys, how="left")
    .merge(plane_info, on="tailnum", how="left")
)
# Plane age in years and the weekday name of each flight
flight_abt["plane_age"] = 2013 - flight_abt["plane_year"]
flight_abt["day_of_week"] = pd.to_datetime(flight_abt[["year", "month", "day"]]).dt.day_name()

# Columns we keep, then a reproducible random sample of 60,000 flights
columns = [
    "month", "day_of_week", "hour", "distance", "carrier", "origin",
    "precip", "visib", "wind_speed", "temp", "plane_age",
    "dep_delay", "arr_delay"
]
classification_data = flight_abt.sample(n=60000, random_state=42)[columns].reset_index(drop=True)

# Put the days in calendar order (Monday first)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
classification_data["day_of_week"] = pd.Categorical(classification_data["day_of_week"], categories=day_order)

classification_data.head()

## 2. Create a Binary Target

The original `arr_delay` column is continuous because it measures minutes.

For classification, we create a binary target:

```python
arr_late_15 = 1 if arr_delay >= 15
arr_late_15 = 0 otherwise
```

This converts the problem from regression to classification.

In [ ]:
# 1 = arrived 15 or more minutes late, 0 = not late
classification_data["arr_late_15"] = (classification_data["arr_delay"] >= 15).astype(int)

classification_data[["arr_delay", "arr_late_15"]].head(10)

In [ ]:
# Number of flights in each class
classification_data["arr_late_15"].value_counts()

In [ ]:
# Share of flights in each class
classification_data["arr_late_15"].value_counts(normalize=True)

### Class balance

About **1 in 4** flights (24.65%) arrived at least 15 minutes late. The classes are **imbalanced**: class 0 is about three times as common as class 1.

This matters: a "model" that always says "not late" would already be correct for about 75% of flights. Keep this in mind when you look at accuracy.

### Your Turn 1

Write one sentence explaining what class 1 and class 0 mean in this lab, and one sentence explaining why the airline might care about the 15-minute line.

**Your answer:**
Type your answer here.

## 3. Define Features and Target

`carrier`, `origin`, and `day_of_week` are text, and `month` is a category (month 12 is not "12 times" month 1). `pd.get_dummies()` turns each category into 0/1 columns, with the first category of each group as the reference.

Model 1 (in-flight alert) uses **all** these columns, including `dep_delay`. We do **not** include `arr_delay` (it defines the target) or `air_time` (known only after landing).

In [ ]:
target = "arr_late_15"

feature_columns = [
    "hour", "distance", "precip", "visib", "wind_speed", "temp", "plane_age",
    "month", "day_of_week", "carrier", "origin", "dep_delay"
]
category_features = ["month", "day_of_week", "carrier", "origin"]

# Turn category columns into 0/1 dummy columns (drop_first drops one reference category)
X = pd.get_dummies(
    classification_data[feature_columns],
    columns=category_features,
    drop_first=True,
    dtype=int
)
y = classification_data[target]

# Model 1: in-flight alert (all columns, including dep_delay)
inflight_columns = list(X.columns)

print("Model 1 uses", len(inflight_columns), "columns")
X.head()

## 4. Train, Validation, and Test Sets

In this lab, we will make choices: which model to prefer and which **threshold** to use. If we made those choices by looking at the test set, the test result would be too optimistic. So we use **three** sets:

- **Training (60%)**: fit the models
- **Validation (20%)**: compare models and choose thresholds
- **Test (20%)**: check the final choice **once**, at the end

We split twice. `stratify` keeps the share of late flights the same in every set. Then we fill missing values (plane age and a few weather values) with the **training** median.

In [ ]:
# First split: 20% test (stratify=y keeps the same late share in every set)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Second split: 25% of the remaining 80% = 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Fill missing values with the TRAINING median
fill_columns = ["precip", "visib", "wind_speed", "temp", "plane_age"]
train_medians = X_train[fill_columns].median()
X_train = X_train.fillna(train_medians)
X_valid = X_valid.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Training:  ", X_train.shape, "late share", round(y_train.mean(), 3))
print("Validation:", X_valid.shape, "late share", round(y_valid.mean(), 3))
print("Test:      ", X_test.shape, "late share", round(y_test.mean(), 3))

## 5. Logistic Regression (Model 1)

Logistic regression is used for classification, even though the name includes "regression." It estimates the **probability** that a flight belongs to class 1 (late).

The features use very different units (minutes, miles, inches of rain), so we first put them on the same scale with `StandardScaler`. `make_pipeline` connects the two steps, so scaling and fitting happen together.

In [ ]:
# Scale the features, then fit logistic regression (max_iter allows enough fitting steps)
log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

log_model.fit(X_train[inflight_columns], y_train)

print("Logistic regression model fitted.")

## 6. Probabilities and the 0.5 Threshold

- `.predict_proba()` gives the predicted **probability** of being late.
- A **threshold** turns the probability into a class. With the usual threshold of 0.5: probability >= 0.5 means "late" (1), otherwise "not late" (0). This is what `.predict()` does.

We evaluate on the **validation** set for now.

In [ ]:
# Column 1 of predict_proba is the probability of being late
log_valid_prob = log_model.predict_proba(X_valid[inflight_columns])[:, 1]
# Flag a flight as late when that probability is 0.5 or higher
log_valid_pred = (log_valid_prob >= 0.5).astype(int)

log_results = pd.DataFrame({
    "actual": y_valid.to_numpy(),
    "predicted_probability_late": log_valid_prob.round(3),
    "predicted_class": log_valid_pred
})

print("Same as .predict():", (log_valid_pred == log_model.predict(X_valid[inflight_columns])).all())
print("Share of flights flagged at 0.5:", round(log_valid_pred.mean(), 3))
print("Share of flights flagged at 0.3:", round((log_valid_prob >= 0.3).mean(), 3))
log_results.head()

A lower threshold flags **more** flights. The threshold is a business choice, not a fixed rule. You will choose one yourself in the offline assignment.

### Coefficients

A positive coefficient means the feature is associated with a **higher predicted probability of being late**, holding the other features constant. It is not automatically causal.

Because we scaled the features, each coefficient describes a change of **one standard deviation** of that feature (for example, about 40 minutes of departure delay), not one minute or one mile. This makes the sizes easier to compare across features.

In [ ]:
# Coefficients from the logistic regression step of the pipeline, largest first
log_coefficients = pd.DataFrame({
    "feature": inflight_columns,
    "coefficient": log_model.named_steps["logisticregression"].coef_[0]
}).sort_values("coefficient", ascending=False)

log_coefficients.head(8).round(3)

### Your Turn 2

Which feature has the largest positive coefficient? Write one cautious interpretation, and explain why "one standard deviation" matters when you compare it with other coefficients.

**Your answer:**
Type your answer here.

## 7. Confusion Matrix and Metrics

A confusion matrix compares actual classes with predicted classes:

| | Predicted 0: not late | Predicted 1: late |
|---|---|---|
| **Actual 0: not late** | True negative | **False alarm** (false positive) |
| **Actual 1: late** | **Missed late flight** (false negative) | True positive (caught) |

In [ ]:
# Rows = actual class, columns = predicted class
cm_log = confusion_matrix(y_valid, log_valid_pred)

cm_log_df = pd.DataFrame(
    cm_log,
    index=["Actual 0: not late", "Actual 1: late"],
    columns=["Predicted 0: not late", "Predicted 1: late"]
)
cm_log_df

### Classification metrics

- **Accuracy:** overall share of correct predictions
- **Precision:** among the flights predicted late, how many were actually late? (Low precision = many false alarms.)
- **Recall:** among the flights that were actually late, how many did the model catch? (Low recall = many missed late flights.)
- **Flag rate:** the share of all flights the model flags as late (how much work the alerts create)

We compare with a **baseline** that always predicts "not late". The helper function below calculates everything the same way for every model.

In [ ]:
def classification_metrics(model_name, dataset, actual, predicted):
    """Return accuracy, precision, recall, flag rate, and error counts as a dictionary."""
    # Unpack the four cells of the 2x2 confusion matrix
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "dataset": dataset,
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted),
        "flag_rate": np.mean(predicted),
        "false_alarms": fp,
        "missed_late": fn,
    }


baseline_valid_pred = np.zeros(len(y_valid), dtype=int)  # always "not late"

pd.DataFrame([
    classification_metrics("Baseline: always not late", "validation", y_valid, baseline_valid_pred),
    classification_metrics("Model 1: logistic regression", "validation", y_valid, log_valid_pred),
]).round(3)

### Why accuracy can mislead

The baseline that **never** flags a flight already reaches about 75% accuracy, but its recall is 0: it catches no late flights at all. So a high accuracy alone does not show that a model is useful. Precision and recall show what kinds of mistakes the model makes.

### Your Turn 3

Write 2-3 sentences interpreting Model 1 (logistic regression) for the operations team. Compare its accuracy with the baseline, and explain its precision and recall in terms of false alarms and missed late flights.

**Your answer:**
Type your answer here.

## 8. Decision Tree Classifier

A decision tree uses a sequence of simple rules to classify flights. It is often easy to explain because it can be drawn as a tree.

To keep the tree readable and to reduce overfitting, we set `max_depth=3` (at most three questions per path). Trees do not need scaled features.

In [ ]:
# A shallow tree (3 levels of questions) stays easy to read
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)

tree_model.fit(X_train[inflight_columns], y_train)

tree_valid_pred = tree_model.predict(X_valid[inflight_columns])

pd.DataFrame([
    classification_metrics("Model 1: decision tree", "validation", y_valid, tree_valid_pred)
]).round(3)

## 9. Visualize the Decision Tree

Each box asks a question about one feature. Flights that answer "yes" go left, flights that answer "no" go right. The bottom boxes give the predicted class.

In [ ]:
# Draw the fitted tree
plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=inflight_columns,
    class_names=["Not late", "Late"],
    filled=False,
    rounded=True,
    impurity=False,
    fontsize=9
)
plt.title("Decision Tree Classifier (Model 1, max_depth=3)")
plt.show()

*Plot description:* The tree's first question is about `dep_delay`: flights with a departure delay of 22.5 minutes or less go left and are mostly predicted "not late", and flights with a larger departure delay go right and are mostly predicted "late". Lower levels ask further questions about departure delay, rain, and distance.

## 10. Compare the Models and Check the Test Set Once

On the validation set, the two Model 1 versions perform similarly. Logistic regression gives probabilities that we can use with any threshold. The tree gives simple rules.

Now we check both models (and the baseline) **once** on the test set, which no model has seen, and save the comparison.

In [ ]:
# Use the test set once: predict with each model, then save the comparison
log_test_pred = log_model.predict(X_test[inflight_columns])
tree_test_pred = tree_model.predict(X_test[inflight_columns])
baseline_test_pred = np.zeros(len(y_test), dtype=int)

classification_comparison = pd.DataFrame([
    classification_metrics("Baseline: always not late", "test", y_test, baseline_test_pred),
    classification_metrics("Model 1: logistic regression", "test", y_test, log_test_pred),
    classification_metrics("Model 1: decision tree", "test", y_test, tree_test_pred),
])

comparison_path = output_dir / "classification_model_comparison_module13.csv"
classification_comparison.to_csv(comparison_path, index=False)
print("Model comparison saved to:", comparison_path)
classification_comparison.round(3)

The test results are close to the validation results, so the validation numbers were a fair guide to performance on new flights.

## 11. Model Suitability

Both models can be useful, but they have different strengths.

**Logistic regression:**

- simple and widely used
- gives probabilities, so the threshold can be adjusted
- gives coefficients
- less intuitive for some audiences

**Decision tree:**

- easy to explain as rules
- can capture non-linear patterns
- can overfit if the tree is too deep
- may be less stable than simpler models

In a business setting, model choice depends on performance, interpretability, and the decision context. For example, an in-flight alert could help the airline contact passengers with tight connections before landing.

## 12. Final Practice: Model 2, the Evening Before

The staffing team must plan **the evening before**. At that time, `dep_delay` is unknown. Build **Model 2** yourself and compare it with Model 1.

In [ ]:
# Final Practice Step 1
# Create evening_columns: every column in inflight_columns except "dep_delay".
# Hint: [col for col in inflight_columns if col != "dep_delay"]

In [ ]:
# Final Practice Step 2
# Create evening_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# and fit it on X_train[evening_columns] and y_train.

In [ ]:
# Final Practice Step 3
# Create evening_valid_prob (probability of late on X_valid[evening_columns])
# and evening_valid_pred = (evening_valid_prob >= 0.5).astype(int).

In [ ]:
# Final Practice Step 4
# Display a DataFrame with two classification_metrics(...) rows on the validation set:
# "Model 1: logistic regression" (log_valid_pred) and "Model 2: evening before" (evening_valid_pred).

**Final Practice answer (2-3 sentences):** How do accuracy, recall, and flag rate of Model 2 compare with Model 1 at the 0.5 threshold? Why is Model 2 so much weaker, and why is the 0.5 threshold a poor choice for it?

Type your answer here.

## 13. Offline Assignment - Evening-Before Staffing Alerts

The operations team will use **Model 2** (the evening-before model from your Final Practice) to flag flights for extra staff and proactive customer messages. The team can review **at most 25% of flights** each day. Within this limit, they want to catch as many late flights as possible.

1. For thresholds 0.20, 0.25, 0.30, ..., 0.50, use `evening_valid_prob` to calculate the **validation** metrics with `classification_metrics()` (use the threshold as the model name, for example `"threshold 0.30"`).
2. Choose the threshold with the **highest recall** whose validation flag rate is **at most 0.25**.
3. Apply **only the chosen threshold** to the **test set** once (`evening_model.predict_proba(X_test[evening_columns])[:, 1]`) and add that test row.
4. Create `threshold_comparison` with all validation rows plus the one test row, and save it as `outputs/classification_threshold_comparison_module13.csv`.
5. Write the recommendation below.

Why validation first? If you chose the threshold by looking at the test set, the test result would no longer be an honest check.

In [ ]:
# Offline Assignment - your code here
# Hints:
# - rows = []
#   for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
#       pred = (evening_valid_prob >= threshold).astype(int)
#       rows.append(classification_metrics(f"threshold {threshold:.2f}", "validation", y_valid, pred))
# - Use the validation flag_rate column to keep only thresholds with flag_rate <= 0.25,
#   then pick the one with the highest recall.
# - Apply only that threshold to the test probabilities and append one "test" row. Name it the same way,
#   for example classification_metrics("threshold 0.35", "test", y_test, test_pred) if you chose 0.35.
# - threshold_comparison = pd.DataFrame(rows); save it with output_dir / the requested filename.

### Staffing Alert Recommendation

**Your response here:** Write 5-6 sentences for the operations manager. State the threshold you chose and why it respects the 25% limit. Report the **test** precision, recall, and flag rate in plain language (for example, "of every 10 flagged flights, about X were actually late"). Explain the trade-off between false alarms and missed late flights at this threshold. Compare briefly with Model 1 and explain why Model 1 cannot be used for this decision. State the main limitation (for example, observed weather instead of a forecast, or 2013 New York data only).

## 14. Applied Practice (Optional, Not Graded): Bank Marketing - Who Should the Call Center Call First?

> **Optional.** This section is not graded, and its output file is not required. It is a good extra practice if your project has a yes/no target: it shows how to rank cases for a limited action list and how to spot a leaky feature.

The flight models showed the workflow and the timing rule. Here you apply them to a problem where the model gives a clear business answer.

**Business scenario.** A Portuguese bank calls clients to offer a term deposit. The call center can make calls to only **10% of the client list** this week. In this practice, the test clients play the role of this week's list. The marketing manager asks: *which clients should we call first, and how much better is that than calling clients at random?*

- **One row:** one client contacted in a campaign (45,211 rows, May 2008 - November 2010)
- **Target:** `subscribed` (yes/no): did the client subscribe to the term deposit?
- **Decision time:** before the call

Data: UCI Machine Learning Repository, *Bank Marketing* dataset (Moro, Rita, and Cortez, 2014), CC BY 4.0. `data/README_data.md` describes the file.

In [ ]:
bank = pd.read_csv(repo_root / "data" / "bank_marketing.csv")
# 1 if the client subscribed, 0 otherwise
bank["subscribed_flag"] = (bank["subscribed"] == "yes").astype(int)

print("Clients:", len(bank))
print("Share who subscribed:", round(bank["subscribed_flag"].mean(), 3))
bank.head()


Only about **12%** of clients subscribed. So if the call center picks clients at random, about 12 of every 100 calls end in a subscription. A useful model must do clearly better than that.

### Timing: which columns are known before the call?

- Known before the call: the client's profile (`age`, `job`, `marital`, `education`, `default`, `balance`, `housing`, `loan`), the contact channel (`contact`), the history with the bank (`pdays`, `previous`, `poutcome`), and the number of calls already made to this client in the current campaign (`campaign`). `pdays = -1` means the client was not contacted in an earlier campaign.
- **Left out on purpose:** `day` and `month` record when the call was made. They are the same for every client on this week's list, so they cannot help choose whom to call. (Subscription rates differ a lot between months in this data, so including them would make the model look better than it is for this decision.)
- **Not known before the call:** `duration`, the length of the call in seconds. It is known only after the call ends. A model with `duration` would look better in the notebook, but it cannot be used to choose whom to call. You will measure this.

The next cell builds the features and a stratified 75%/25% train/test split. No threshold is chosen on the test set here: the business rule (call the top 10%) is fixed in advance, so a separate validation set is not needed.

In [ ]:
precall_features = [
    "age", "job", "marital", "education", "default", "balance", "housing", "loan",
    "contact", "campaign", "pdays", "previous", "poutcome",
]

# Dummy columns for every text column
X_bank = pd.get_dummies(bank[precall_features + ["duration"]], drop_first=True, dtype=int)
y_bank = bank["subscribed_flag"]

# Before the call: every column except duration
precall_columns = [col for col in X_bank.columns if col != "duration"]
# After the call (leakage): the same columns plus duration
aftercall_columns = precall_columns + ["duration"]

# 75% training, 25% test (stratify keeps the same subscriber share in both)
X_bank_train, X_bank_test, y_bank_train, y_bank_test = train_test_split(
    X_bank, y_bank, test_size=0.25, random_state=42, stratify=y_bank
)
print("Training clients:", len(X_bank_train), "| test clients:", len(X_bank_test))


### A helper for a call list

`call_list_metrics()` ranks the test clients by predicted probability, "calls" the top 10%, and reports:

- **precision:** the share of called clients who subscribed;
- **recall:** the share of all subscribers in the test set who were called;
- **lift:** precision divided by the overall subscription rate (how many times better than random calling). Random calling should give a lift close to 1.

In [ ]:
def call_list_metrics(model_name, actual, probability, share=0.10):
    """Call the top share of clients by predicted probability and report precision, recall, and lift."""
    actual = np.asarray(actual)
    n_calls = int(round(share * len(actual)))
    # Mark the n_calls clients with the highest probability as called (1)
    called = np.zeros(len(actual), dtype=int)
    called[np.argsort(-np.asarray(probability))[:n_calls]] = 1
    # Precision = share of calls that reach a subscriber; lift compares it with random calling
    precision = precision_score(actual, called)
    return {
        "model": model_name,
        "calls": n_calls,
        "subscribers_reached": int(actual[called == 1].sum()),
        "precision": precision,
        "recall": recall_score(actual, called),
        "lift_vs_random": precision / actual.mean(),
    }


In [ ]:
# Applied Practice Step 1
# Create campaign_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
# and fit it on X_bank_train[precall_columns] and y_bank_train.
# Then create campaign_prob: the predicted probability of subscribing for X_bank_test[precall_columns].
# Hint: campaign_model.predict_proba(...)[:, 1]


In [ ]:
# Applied Practice Step 2
# Fit leaky_model the same way on aftercall_columns (with duration) and create leaky_prob for the test clients.
# Then create campaign_comparison, a DataFrame with three rows:
#   1. call_list_metrics("Random calling", y_bank_test, random_prob) where
#      random_prob = np.random.default_rng(42).random(len(y_bank_test))
#   2. call_list_metrics("Before the call (no duration)", y_bank_test, campaign_prob)
#   3. call_list_metrics("After the call (with duration, leakage)", y_bank_test, leaky_prob)
# Display campaign_comparison.round(3) and save it:
#   campaign_comparison.to_csv(output_dir / "bank_campaign_comparison_module13.csv", index=False)


### What drives the ranking?

The next cell shows the subscription rate by the outcome of the **previous** campaign (`poutcome`) and, after Step 1, the five largest and five smallest coefficients of `campaign_model`.

**How to read these coefficients.** All columns were scaled, including the 0/1 columns, so each coefficient describes a change of one standard deviation of that column. Use the coefficients to see the **direction** (positive: higher predicted chance of subscribing) and the **rough order** of influence, not as "the effect of being in a category". For 0/1 columns, the comparison is with the reference category, which is the first value in alphabetical order (for example, `poutcome` is compared with `failure`, and `contact` with `cellular`).

In [ ]:
# Applied Practice Step 3 (given)
# Number of clients and subscription rate for each previous-campaign outcome
previous_outcome = (
    bank.groupby("poutcome")["subscribed_flag"]
    .agg(clients="size", subscription_rate="mean")
    .round(3)
    .sort_values("subscription_rate", ascending=False)
)
display(previous_outcome)

if "campaign_model" not in globals():
    print("Complete Applied Practice Step 1 first, then run this cell again.")
else:
    # Pre-call model coefficients, smallest to largest
    bank_coefficients = pd.Series(
        campaign_model.named_steps["logisticregression"].coef_[0], index=precall_columns
    ).sort_values()
    # Show the 5 largest positive, then the 5 most negative coefficients
    display(pd.concat([bank_coefficients.tail(5)[::-1], bank_coefficients.head(5)]).round(2).to_frame("coefficient"))


### Campaign Recommendation (Optional)

**Your response here:** Write 5-6 sentences for the marketing manager.

1. Compare the before-the-call model with random calling: of every 100 calls, about how many end in a subscription, and what is the lift?
2. What share of all subscribers does the 10% call list reach?
3. Which kind of client should be called first, based on the subscription rates and the coefficients? Use cautious (non-causal) language. (For example, `campaign` is lower for subscribers partly because the bank stops calling a client after they subscribe.)
4. Explain why the model with `duration` cannot be used for this decision, even though its numbers are better.
5. State one limitation, for example the data period (2008-2010) or one country.

## 15. Final Reflection

Write 4-5 sentences answering:

1. How is classification different from regression?
2. What was the target variable, and what does one row represent?
3. What did the confusion matrix show about false alarms and missed late flights?
4. Why can accuracy be misleading in this lab?
5. Which model seemed more interpretable, and why?
6. How might classification (and the timing check) apply to your project?

**Your reflection:**
Type your answer here.

## 16. Save Your Work

Week 12 and Week 13 use the same repository. **Week 13 is graded on the Module 13 deadline from this module13 notebook and its two required Module 13 output files.** (Week 12 was graded on the Module 12 deadline from the module12 notebook.)

Before the Module 13 deadline:

1. Complete all Your Turn, Final Practice, and Offline Assignment sections. The Applied Practice (Section 14) is optional.
2. Complete the staffing alert recommendation and the final reflection.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm both required Module 13 CSV files exist in `outputs`:
   - `classification_model_comparison_module13.csv`
   - `classification_threshold_comparison_module13.csv`

   If you did the optional Applied Practice, `bank_campaign_comparison_module13.csv` is also there; it is not required.
5. Save the notebook, then commit and push your work.

```bash
git add .
git commit -m "Complete Module 13 classification lab"
git push
```